# Fine-Tune XLM-RoBERTa for Text Classification

This notebook fine-tunes [xlm-roberta-base](https://huggingface.co/xlm-roberta-base) (or `xlm-roberta-large`)
for **text classification** — works for both single-text and text-pair inputs in any language.

## What you need
- A **CSV file** with a `text` column and a `label` column
- Optionally a `text_pair` column for sentence-pair tasks (e.g., NLI, similarity, QE)
- A **GPU runtime** (Colab T4 or better)

## How to customize
1. Upload your CSV and set `data_path` in the config
2. Set your label names in `label_names` (e.g., `["positive", "negative"]`)
3. Adjust hyperparameters as needed
4. Run all cells

## Example use cases
- Sentiment analysis (single text → positive/negative)
- Toxicity detection (single text → toxic/not_toxic)
- Natural language inference (text pair → entailment/neutral/contradiction)
- Quality estimation (source + translation → good/bad)

## 1. Install Dependencies

In [ ]:
!pip install -q transformers torch pandas scikit-learn evaluate

## 2. Imports & GPU Setup

In [ ]:
import json
import os
import warnings
from dataclasses import asdict, dataclass
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

if IN_COLAB:
    drive.mount("/content/drive")

## 3. Configuration

**Edit the fields below to match your task.** At minimum, set:
- `data_path` — path to your CSV
- `label_names` — list of class names matching your `label` column values
- `text_pair_column` — set to your second text column name for pair tasks, or `""` for single-text

In [ ]:
@dataclass
class Config:
    # === Data ===
    data_path: str = "/content/drive/MyDrive/data/classification_data.csv"
    text_column: str = "text"           # column with input text
    text_pair_column: str = ""          # column with second text (leave empty for single-text tasks)
    label_column: str = "label"         # column with class labels
    label_names: Tuple[str, ...] = ("negative", "positive")  # class names in order
    output_dir: str = "/content/drive/MyDrive/xlmr-classifier-output"
    test_size: float = 0.1             # fraction held out for val + test
    split_seed: int = 42

    # === Model ===
    base_model: str = "xlm-roberta-base"  # or "xlm-roberta-large"
    max_length: int = 512

    # === Training ===
    num_epochs: int = 8
    train_batch_size: int = 32
    eval_batch_size: int = 64
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    logging_steps: int = 50
    early_stopping_patience: int = 3
    seed: int = 42

    # === Class balancing ===
    oversample_minority: bool = False   # oversample minority class(es) to balance training
    use_class_weights: bool = False     # use inverse-frequency class weights in loss


CONFIG = Config()

label2id = {name: i for i, name in enumerate(CONFIG.label_names)}
id2label = {i: name for i, name in enumerate(CONFIG.label_names)}
num_labels = len(CONFIG.label_names)

print(json.dumps(asdict(CONFIG), indent=2))
print(f"\nLabels: {label2id}")

## 4. Load & Split Data

In [ ]:
def load_and_split_data(cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    df = pd.read_csv(cfg.data_path)
    assert cfg.text_column in df.columns, f"CSV must have '{cfg.text_column}' column"
    assert cfg.label_column in df.columns, f"CSV must have '{cfg.label_column}' column"
    if cfg.text_pair_column:
        assert cfg.text_pair_column in df.columns, f"CSV must have '{cfg.text_pair_column}' column"

    # Clean
    df = df.dropna(subset=[cfg.text_column, cfg.label_column])
    df = df[df[cfg.text_column].astype(str).str.strip() != ""]
    if cfg.text_pair_column:
        df = df.dropna(subset=[cfg.text_pair_column])
        df = df[df[cfg.text_pair_column].astype(str).str.strip() != ""]
    df = df.reset_index(drop=True)

    # Validate labels
    unique_labels = set(df[cfg.label_column].unique())
    expected_labels = set(cfg.label_names)
    unexpected = unique_labels - expected_labels
    if unexpected:
        print(f"WARNING: Found unexpected labels: {unexpected} — these rows will be dropped")
        df = df[df[cfg.label_column].isin(expected_labels)].reset_index(drop=True)

    print(f"Loaded {len(df)} valid examples")
    print(f"Label distribution:")
    for name in cfg.label_names:
        count = (df[cfg.label_column] == name).sum()
        print(f"  {name}: {count} ({100 * count / len(df):.1f}%)")

    # Split
    train_df, temp_df = train_test_split(df, test_size=cfg.test_size * 2, random_state=cfg.split_seed, stratify=df[cfg.label_column])
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=cfg.split_seed, stratify=temp_df[cfg.label_column])

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    print(f"Split: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")
    return train_df, val_df, test_df


train_df, val_df, test_df = load_and_split_data(CONFIG)

## 5. Balance Training Data (Optional)

In [ ]:
train_labels = [label2id[l] for l in train_df[CONFIG.label_column]]

if CONFIG.oversample_minority:
    max_count = max(train_df[CONFIG.label_column].value_counts())
    balanced_parts = []
    for name in CONFIG.label_names:
        subset = train_df[train_df[CONFIG.label_column] == name]
        if len(subset) < max_count:
            subset = subset.sample(n=max_count, replace=True, random_state=CONFIG.split_seed)
        balanced_parts.append(subset)
    train_df = pd.concat(balanced_parts, ignore_index=True)
    train_df = train_df.sample(frac=1.0, random_state=CONFIG.split_seed).reset_index(drop=True)
    train_labels = [label2id[l] for l in train_df[CONFIG.label_column]]
    print(f"After oversampling: {len(train_df)} samples (balanced)")

val_labels = [label2id[l] for l in val_df[CONFIG.label_column]]
test_labels = [label2id[l] for l in test_df[CONFIG.label_column]]

# Compute class weights if needed
class_weights = None
if CONFIG.use_class_weights:
    counts = np.bincount(train_labels, minlength=num_labels).astype(float)
    class_weights = torch.tensor(len(train_labels) / (num_labels * counts), dtype=torch.float32)
    if torch.cuda.is_available():
        class_weights = class_weights.cuda()
    print(f"Class weights: {dict(zip(CONFIG.label_names, class_weights.tolist()))}")

## 6. Tokenize

In [ ]:
class TextClassificationDataset(Dataset):
    """Tokenized dataset for single-text or text-pair classification."""
    def __init__(self, texts, labels, tokenizer, max_length, text_pairs=None):
        if text_pairs is not None:
            self.encodings = tokenizer(
                texts, text_pairs,
                padding=True, truncation=True, max_length=max_length,
                return_tensors="pt",
            )
        else:
            self.encodings = tokenizer(
                texts,
                padding=True, truncation=True, max_length=max_length,
                return_tensors="pt",
            )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


def make_dataset(df: pd.DataFrame, labels: List[int], tokenizer, cfg: Config):
    texts = df[cfg.text_column].fillna("").astype(str).tolist()
    text_pairs = None
    if cfg.text_pair_column:
        text_pairs = df[cfg.text_pair_column].fillna("").astype(str).tolist()
    return TextClassificationDataset(texts, labels, tokenizer, cfg.max_length, text_pairs)


print(f"Loading tokenizer: {CONFIG.base_model}")
tokenizer = AutoTokenizer.from_pretrained(CONFIG.base_model)

print("Tokenizing datasets...")
train_dataset = make_dataset(train_df, train_labels, tokenizer, CONFIG)
val_dataset = make_dataset(val_df, val_labels, tokenizer, CONFIG)
test_dataset = make_dataset(test_df, test_labels, tokenizer, CONFIG)
print(f"Done: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")

## 7. Load Model

In [ ]:
print(f"Loading model: {CONFIG.base_model} ({num_labels} labels)")
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG.base_model,
    num_labels=num_labels,
    problem_type="single_label_classification",
    id2label=id2label,
    label2id=label2id,
)

## 8. Training

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")
    f1_weighted = f1_score(labels, preds, average="weighted")

    metrics = {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
    }

    # Per-class F1
    for i, name in id2label.items():
        metrics[f"f1_{name}"] = f1_score(labels, preds, average="binary", pos_label=i, zero_division=0)

    # AUROC for binary classification
    if num_labels == 2:
        probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()
        try:
            metrics["auroc"] = roc_auc_score(labels, probs[:, 1])
        except ValueError:
            metrics["auroc"] = 0.5

    return metrics


class WeightedTrainer(Trainer):
    """Trainer with optional class-weighted loss."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        if class_weights is not None:
            loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
        else:
            loss_fn = torch.nn.CrossEntropyLoss()
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
os.makedirs(CONFIG.output_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=os.path.join(CONFIG.output_dir, "checkpoints"),
    num_train_epochs=CONFIG.num_epochs,
    per_device_train_batch_size=CONFIG.train_batch_size,
    per_device_eval_batch_size=CONFIG.eval_batch_size,
    learning_rate=CONFIG.learning_rate,
    weight_decay=CONFIG.weight_decay,
    warmup_ratio=CONFIG.warmup_ratio,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=CONFIG.logging_steps,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=CONFIG.seed,
)

TrainerClass = WeightedTrainer if CONFIG.use_class_weights else Trainer
trainer = TrainerClass(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=CONFIG.early_stopping_patience)],
)

trainer.train()

## 9. Save Model

In [ ]:
model_dir = os.path.join(CONFIG.output_dir, "best_model")
trainer.save_model(model_dir)
tokenizer.save_pretrained(model_dir)
print(f"Model saved to: {model_dir}")

## 10. Evaluate on Test Set

In [ ]:
print("Evaluating on test set...")
test_metrics = trainer.evaluate(test_dataset)

print(f"\nTest Results ({len(test_df)} samples)")
print(f"{'='*50}")
print(f"  Accuracy:    {test_metrics['eval_accuracy']:.4f}")
print(f"  F1 (macro):  {test_metrics['eval_f1_macro']:.4f}")
print(f"  F1 (weighted): {test_metrics['eval_f1_weighted']:.4f}")
for name in CONFIG.label_names:
    print(f"  F1 ({name}): {test_metrics[f'eval_f1_{name}']:.4f}")
if num_labels == 2:
    print(f"  AUROC:       {test_metrics['eval_auroc']:.4f}")

# Get predictions
predictions = trainer.predict(test_dataset)
logits = predictions.predictions
pred_labels = np.argmax(logits, axis=-1)
probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()

# Classification report
print(f"\n{'='*50}")
print("Classification Report")
print("=" * 50)
print(classification_report(
    test_labels, pred_labels,
    labels=list(range(num_labels)),
    target_names=list(CONFIG.label_names),
    digits=4,
))

# Confusion matrix
print(f"{'='*50}")
print("Confusion Matrix (rows=true, cols=predicted)")
print("=" * 50)
cm = confusion_matrix(test_labels, pred_labels, labels=list(range(num_labels)))
header = f"{'':>14}" + "".join(f"{n:>14}" for n in CONFIG.label_names)
print(header)
for i, name in enumerate(CONFIG.label_names):
    row = f"{name:>14}" + "".join(f"{cm[i, j]:>14}" for j in range(num_labels))
    print(row)

## 11. Save Predictions & Results

In [ ]:
# Add predictions to test dataframe
results_df = test_df.copy()
results_df["predicted_label"] = [id2label[p] for p in pred_labels]
for i, name in enumerate(CONFIG.label_names):
    results_df[f"prob_{name}"] = probs[:, i]

results_df.to_csv(os.path.join(CONFIG.output_dir, "test_predictions.csv"), index=False)

# Save metrics
results = {
    "config": {k: v for k, v in asdict(CONFIG).items()},
    "n_train": len(train_df),
    "n_val": len(val_df),
    "n_test": len(test_df),
    "test_metrics": {k.replace("eval_", ""): float(v) for k, v in test_metrics.items() if isinstance(v, (int, float))},
    "confusion_matrix": cm.tolist(),
}
with open(os.path.join(CONFIG.output_dir, "results.json"), "w") as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved to: {CONFIG.output_dir}")
print(f"  test_predictions.csv — per-sample predictions with probabilities")
print(f"  results.json — metrics and config")
print(f"  best_model/ — fine-tuned model weights")

## 12. Inference Example

Use the fine-tuned model to classify new text:

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=model_dir,
    tokenizer=model_dir,
    device=0 if torch.cuda.is_available() else -1,
    top_k=None,  # return all class probabilities
)

# Single-text examples (adjust to your task)
samples = [
    "This product is amazing, I love it!",
    "Terrible experience, would not recommend.",
]

for text in samples:
    result = classifier(text)[0]
    top = max(result, key=lambda x: x["score"])
    print(f"  Text: {text[:80]}")
    print(f"  Prediction: {top['label']} ({top['score']:.3f})")
    print()